# Exploration OULAD

Ce notebook a pour but d'explorer les données issues du **Open University Learning Analystic Dataset** (OULAD).

Cette analyse de données contitue une étape essentielle avant la mise en place des modèles de **théorie de la réponse aux items (IRT)**, cela permettra de comprendre la structure des variables, la nature des intéractions étudiants-évualation ainsi que la qualité des réponses.


Ici, on charges les différents fichiers du datasets, on analyse leur structures, on identifie les variables pértinentes pour la modélisation IRT et enfin on fera une **matrice étudiant x item** exploitable par les modèles.

Cette étape est fondamentale pour la suite du projet visant à implémenter plusieurs modèles de la théories de la réponse aux itemps allant de **1PL à 4PL**.

In [13]:
import pandas as pd
from pandas.core.reshape.pivot import pivot_table

On a importé **pandas** afin de charger les datasets et de transformer les données. Le "pivot_table" sert à structurer le tableau pour avoir un étudiant par ligne et un item par colonne pour avoir cette matrice étudiant x item.


Dans un contexte réel, les notes sont continues, ce qui implique l'utilisation d'un indicateur binaire pour binomiser les réponses, par exemple : 

$$X_{ij} = \mathbb{1}_{{x}_{ij} \le 50\%}$$

In [ ]:
df_student_asmnt = pd.read_csv("../data/studentAssessment.csv")
df_asmnt = pd.read_csv("../data/assessments.csv")
df_merged = df_student_asmnt.merge(df_asmnt, on = "id_assessment") #on fusionne pour savoir à quel item appartient à quel module
df_filtered = df_merged[(df_merged['code_module'] == 'BBB') & 
                        (df_merged['code_presentation'] == '2013B')]


matrix_raw = df_filtered.pivot_table(
    index = "id_student",
    columns = "id_assessment",
    values = "score"
)

matrix  = (matrix>50).astype(int) #si la valeur dépasse 50 alors on considère que l'etudiant a reussi son activité donc 1 (sinon 0)
print(matrix.head())

id_assessment  1752   1753   1754   1755   ...  37440  37441  37442  37443
id_student                                 ...                            
6516               0      0      0      0  ...      0      0      0      0
8462               0      0      0      0  ...      0      0      0      0
11391              0      0      0      0  ...      0      0      0      0
23629              0      0      0      0  ...      0      0      0      0
23698              0      0      0      0  ...      0      0      0      0

[5 rows x 188 columns]


In [ ]:
import numpy as np
matrix_with_nan = df.pivot_table(index="id_student", columns="id_assessment", values="score")
mask = ~matrix_with_nan.isna()
M = mask.astype(int)
X = (matrix_with_nan > 40).fillna(0).astype(int)
print(M.head())

# Affiche un extrait du masque
print("Extrait du masque M (1 = présent, 0 = absent) :")
print(M.iloc[:5, :5]) 

id_assessment  1752   1753   1754   1755   ...  37440  37441  37442  37443
id_student                                 ...                            
6516               0      0      0      0  ...      0      0      0      0
8462               0      0      0      0  ...      0      0      0      0
11391              1      1      1      1  ...      0      0      0      0
23629              0      0      0      0  ...      0      0      0      0
23698              0      0      0      0  ...      0      0      0      0

[5 rows x 188 columns]
Extrait du masque M (1 = présent, 0 = absent) :
id_assessment  1752  1753  1754  1755  1756
id_student                                 
6516              0     0     0     0     0
8462              0     0     0     0     0
11391             1     1     1     1     1
23629             0     0     0     0     0
23698             0     0     0     0     0


# Synthèse de l'échantillon
Analysons le nombre d'étudiants, d'items et le nombre total des données ignorées (NaN)

In [21]:
print(f"\nNombre total de données ignorées (NaN) : {(M == 0).sum().sum()}")
print(f"Nombre d'étudiants (lignes) : {matrix.shape[0]}")
print(f"Nombre d'items/examens (colonnes) : {matrix.shape[1]}")


Nombre total de données ignorées (NaN) : 4216249
Nombre d'étudiants (lignes) : 23351
Nombre d'items/examens (colonnes) : 188


# Gestion des données manquantes

Le dataset OULAD est très sparse. Lorsqu'un étudiant n'a pas répondu à une question d'examen, cela affiche la valeur "NaN", signifiant Not a Number, au lieu d'afficher "0". 
Le soucis est que le NaN a un "effet propageur", il rend les calculs matriciels et l'optimiastion des gradients impossibles. 
On pourrait "nettoyer" les NaN en les remplaçant par 0, cependant cela reviendrait à traiter une absence comme un échec, ce qui ferait chuter artificiellement le niveau estimé de l'étudiant et fausserait l'IRT.

Pour éviter de biaiser l'IRT, on implémente une matrice de masquage M: 
- $M_i$,$_j$ = 1 si l'étudiant $i$ a répondu à l'item $j$
- $M_i$,$_j$ = 0 si l'étudiant $i$ n'a pas répondu à l'item $j$

Ce masquage permettra d'ignorer les "0" des données manquantes lors du calcul de la fonction de perte (Loss) et la descente du gradient en mettant à jour le modèle que sur des réponses réelles.